# Notebook 05: Spins, Phases, and Collective Behavior

**QCCCM** — Quantum Compute for Computational Cognitive Modeling

---

## What you'll learn

1. The **Ising model** as the simplest model of collective behavior
2. **Phase transitions**: how a system qualitatively changes at a critical temperature
3. **Frustration**: when agents can't all be satisfied simultaneously
4. **Multi-agent agreement** as a spin system (connecting to Nayebi, 2025)
5. How **quantum fluctuations** (transverse field) help escape frustrated states
6. The mathematical scaffold for understanding the minority game (NB 04)

### Why this matters for cognitive science

The Ising model is the "hydrogen atom" of collective behavior. It teaches three concepts that recur everywhere in multi-agent cognitive modeling:

- **Phase transitions** — sudden qualitative regime changes (opinion shifts, market crashes, neural criticality)
- **Frustration** — conflicting objectives that prevent global satisfaction (alignment, social dilemmas)
- **Order parameters** — macroscopic quantities that summarise the collective state (consensus, polarisation)

These are the same concepts that drive the minority game (NB 04), multi-agent agreement complexity (Nayebi, 2025), and quantum-enhanced collective decision-making.

### Historical context

The connection between game theory and statistical mechanics runs deep:

| Year | Development |
|------|-------------|
| 1920 | Ising introduces the spin model of ferromagnetism |
| 1944 | Onsager solves the 2D Ising model exactly |
| 1973 | Schelling's segregation model (equivalent to a lattice gas) |
| 1994 | Brock & Durlauf: discrete choice as Ising with social interactions |
| 1997 | Challet & Zhang: minority game (frustrated anti-ferromagnet) |
| 2000 | Marsili, Challet, Zecchina: replica analysis of minority game |
| 2005 | Bouchaud: wealth condensation as Bose-Einstein condensation |
| 2010 | Khrennikov: quantum-like models from classical stat mech |
| 2025 | Nayebi: information-theoretic bounds on multi-agent agreement |

The common thread: **when many agents interact under constraints, statistical mechanics provides the natural mathematical language**, whether the agents are atoms, neurons, or AI systems.

### Prerequisites
- NB 01 (density matrices, entropy)
- Linear algebra, basic probability
- No physics background required — we build everything from scratch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qcccm.games.agreement import (
    IsingParams, run_ising, phase_diagram,
    ising_energy, ising_magnetisation,
    AgreementParams, run_agreement_simulation, agreement_scaling,
)

plt.rcParams['figure.dpi'] = 120
print("Ising model and agreement modules loaded")

## 1. The Ising Model: Spins on a Grid

Imagine a grid of agents, each holding a binary opinion: +1 or −1. Each agent prefers to **agree with its neighbours** (ferromagnetic coupling). The total "energy" of the system is:

$$H = -J \sum_{\langle i,j \rangle} s_i s_j - h \sum_i s_i$$

- $J > 0$: agents prefer alignment (ferromagnetic)
- $J < 0$: agents prefer disagreement (anti-ferromagnetic → frustration!)
- $h$: external bias (like a news signal pushing everyone one way)
- $T$: temperature = noise = bounded rationality

At **low T**: agents lock into consensus (ordered phase, $|m| \approx 1$)
At **high T**: noise overwhelms coupling (disordered phase, $|m| \approx 0$)
At **$T_c \approx 2.269$**: the phase transition — the system is poised between order and disorder

In [ ]:
# Visualise spin configurations at different temperatures
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
temps = [1.0, 2.27, 4.0]
labels = ['Ordered (T=1.0)', 'Critical (T≈2.27)', 'Disordered (T=4.0)']

for ax, T, label in zip(axes, temps, labels):
    params = IsingParams(L=30, temperature=T, n_steps=500)
    _, _, spins = run_ising(params, seed=42)
    ax.imshow(spins, cmap='RdBu', vmin=-1, vmax=1, interpolation='nearest')
    ax.set_title(label, fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])
    m = ising_magnetisation(spins)
    ax.text(1, 28, f'|m|={m:.2f}', color='white', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

plt.suptitle('Ising Model: Order → Criticality → Disorder', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Time evolution at the three temperatures
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, T, label in zip(axes, [1.0, 2.27, 4.0], ['T=1.0', 'T=2.27', 'T=4.0']):
    params = IsingParams(L=20, temperature=T, n_steps=500)
    energies, mags, _ = run_ising(params, seed=42)
    ax.plot(mags, 'b-', alpha=0.7, linewidth=0.8)
    ax.set_xlabel('MC Step')
    ax.set_ylabel('|m|')
    ax.set_title(f'{label}: magnetisation over time')
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 2. The Phase Transition

The order parameter $|m|$ (magnetisation = consensus) drops from ~1 to ~0 as temperature increases through $T_c$. The **susceptibility** $\chi = N \cdot \text{Var}(m) / T$ peaks at the transition — the system is maximally sensitive to perturbation.

This is the same structure as:
- **Minority game**: σ²/N peaks at α_c (NB 04)
- **Neural criticality**: cortical dynamics show signatures of critical Ising-like transitions
- **Opinion dynamics**: sudden opinion shifts in social networks

In [ ]:
# Phase diagram
T, mag, energy, chi = phase_diagram(
    L=20, temperatures=np.linspace(1.0, 4.0, 30),
    n_steps=800, n_equilibrate=300, seed=42,
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(T, mag, 'bo-', markersize=3, linewidth=1.5)
axes[0].axvline(x=2.269, color='red', linestyle='--', alpha=0.7, label='$T_c$ = 2.269')
axes[0].set_xlabel('Temperature T')
axes[0].set_ylabel('|m| (consensus)')
axes[0].set_title('Order Parameter')
axes[0].legend()

axes[1].plot(T, energy, 'go-', markersize=3, linewidth=1.5)
axes[1].axvline(x=2.269, color='red', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Temperature T')
axes[1].set_ylabel('E/N')
axes[1].set_title('Energy per Spin')

axes[2].plot(T, chi, 'ro-', markersize=3, linewidth=1.5)
axes[2].axvline(x=2.269, color='red', linestyle='--', alpha=0.7)
axes[2].set_xlabel('Temperature T')
axes[2].set_ylabel('χ (susceptibility)')
axes[2].set_title('Susceptibility (peaks at $T_c$)')

plt.suptitle('2D Ising Phase Diagram (L=20)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Frustration: When Agents Can't All Agree

In the standard Ising model ($J > 0$), all spins *want* to align — there's no conflict. But what if some bonds are **anti-ferromagnetic** ($J < 0$)? Now neighbouring agents want to *disagree*.

On a triangle with two ferromagnetic bonds and one anti-ferromagnetic bond, at least one bond is always frustrated — there is no configuration that satisfies everyone. This is the essence of:

- **Spin glass physics** (Sherrington-Kirkpatrick model)
- **Social dilemmas** (can't satisfy all relationships)
- **AI alignment** (conflicting objectives — Nayebi's framework)
- **Minority game** (everyone can't be in the minority)

In [ ]:
# Demonstrate frustration with the agreement simulation
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# No frustration
params_easy = AgreementParams(n_agents=20, n_objectives=5, frustration=0.0, max_rounds=300, seed=42)
result_easy = run_agreement_simulation(params_easy)
axes[0].plot(result_easy.magnetisation_trajectory, 'b-', linewidth=1.5)
axes[0].set_xlabel('Round')
axes[0].set_ylabel('Consensus |m|')
axes[0].set_title(f'No frustration\nFinal disagreement: {result_easy.final_disagreement:.2f}')
axes[0].set_ylim(0, 1.05)
axes[0].axhline(y=0.9, color='red', linestyle='--', alpha=0.5, label='ε threshold')
axes[0].legend()

# 40% frustrated bonds
params_hard = AgreementParams(n_agents=20, n_objectives=5, frustration=0.4, max_rounds=300, seed=42)
result_hard = run_agreement_simulation(params_hard)
axes[1].plot(result_hard.magnetisation_trajectory, 'r-', linewidth=1.5)
axes[1].set_xlabel('Round')
axes[1].set_ylabel('Consensus |m|')
axes[1].set_title(f'40% frustration\nFinal disagreement: {result_hard.final_disagreement:.2f}')
axes[1].set_ylim(0, 1.05)
axes[1].axhline(y=0.9, color='red', linestyle='--', alpha=0.5, label='ε threshold')
axes[1].legend()

plt.suptitle('Frustration Prevents Consensus', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Multi-Agent Agreement and Nayebi's Bounds

Nayebi (2025) proved that N agents reaching ε-agreement on M objectives requires **at least Ω(MN² log(1/ε))** bits of communication — no algorithm can do better.

We can simulate this: agents on a fully-connected graph exchange messages and update beliefs via Metropolis-like dynamics. The question: **does the empirical communication cost match the N² scaling?**

In [ ]:
# Communication cost scaling: classical agents
agent_counts_c, msgs_c_mean, msgs_c_std = agreement_scaling(
    agent_counts=[5, 8, 12, 16, 20, 25],
    n_objectives=5, epsilon=0.1, temperature=1.0,
    frustration=0.0, quantumness=0.0,
    max_rounds=300, n_seeds=5,
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(agent_counts_c, msgs_c_mean, yerr=msgs_c_std, fmt='bo-',
            linewidth=2, markersize=6, capsize=4, label='Classical')

# Fit N^alpha
from numpy.polynomial import polynomial as P
log_n = np.log(agent_counts_c)
log_m = np.log(msgs_c_mean)
mask = np.isfinite(log_m)
if mask.sum() > 1:
    coeffs = np.polyfit(log_n[mask], log_m[mask], 1)
    alpha = coeffs[0]
    n_fit = np.linspace(agent_counts_c[0], agent_counts_c[-1], 50)
    ax.plot(n_fit, np.exp(np.polyval(coeffs, np.log(n_fit))), 'b--',
            alpha=0.5, label=f'Fit: N^{alpha:.1f}')

ax.set_xlabel('Number of Agents N')
ax.set_ylabel('Messages to Agreement')
ax.set_title('Communication Cost Scaling (Nayebi predicts Ω(N²))')
ax.legend()
ax.set_xscale('log')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print(f"Empirical scaling exponent: {alpha:.2f}")
print(f"Nayebi lower bound predicts: 2.0 (quadratic in N)")

## 5. Quantum Fluctuations: Tunneling Through Frustration

In the quantum Ising model, a **transverse field** $\Gamma$ adds quantum fluctuations that allow the system to tunnel between spin configurations without thermally climbing over energy barriers.

For multi-agent agreement, this translates to: quantum coherence lets agents explore the space of possible agreements more efficiently, potentially reducing the communication overhead.

The key question: **does quantum-enhanced agreement change the N² scaling exponent?**

In [ ]:
# Compare classical vs quantum agreement scaling
agent_counts_q, msgs_q_mean, msgs_q_std = agreement_scaling(
    agent_counts=[5, 8, 12, 16, 20, 25],
    n_objectives=5, epsilon=0.1, temperature=1.0,
    frustration=0.0, quantumness=0.3,
    max_rounds=300, n_seeds=5,
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(agent_counts_c, msgs_c_mean, yerr=msgs_c_std, fmt='bo-',
            linewidth=2, markersize=6, capsize=4, label='Classical')
ax.errorbar(agent_counts_q, msgs_q_mean, yerr=msgs_q_std, fmt='gs-',
            linewidth=2, markersize=6, capsize=4, label='Quantum (q=0.3)')

# Fit both
for counts, means, color, label in [
    (agent_counts_c, msgs_c_mean, 'blue', 'Classical'),
    (agent_counts_q, msgs_q_mean, 'green', 'Quantum'),
]:
    log_n = np.log(counts)
    log_m = np.log(means)
    mask = np.isfinite(log_m)
    if mask.sum() > 1:
        coeffs = np.polyfit(log_n[mask], log_m[mask], 1)
        n_fit = np.linspace(counts[0], counts[-1], 50)
        ax.plot(n_fit, np.exp(np.polyval(coeffs, np.log(n_fit))), '--',
                color=color, alpha=0.5, label=f'{label}: N^{coeffs[0]:.1f}')

ax.set_xlabel('Number of Agents N')
ax.set_ylabel('Messages to Agreement')
ax.set_title('Classical vs Quantum Agreement Scaling')
ax.legend()
ax.set_xscale('log')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# Frustrated agreement: where quantum effects should help most
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, frust, title in zip(axes, [0.0, 0.3], ['No Frustration', '30% Frustration']):
    for q, color, label in [(0.0, 'blue', 'Classical'), (0.3, 'green', 'Quantum (q=0.3)')]:
        params = AgreementParams(
            n_agents=20, n_objectives=5, frustration=frust,
            temperature=1.0, max_rounds=300, seed=42,
        )
        result = run_agreement_simulation(params, quantumness=q)
        ax.plot(result.magnetisation_trajectory, color=color, linewidth=1.5,
                label=f'{label} (disagree={result.final_disagreement:.2f})')

    ax.set_xlabel('Round')
    ax.set_ylabel('Consensus |m|')
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)

plt.suptitle('Quantum Tunneling Helps Most Under Frustration', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 6. Connection Map

| Concept | Ising Model | Minority Game (NB 04) | Agreement (Nayebi) | QCCCM Module |
|---|---|---|---|---|
| Agent | Spin $s_i = \pm 1$ | Player choosing 0/1 | Bayesian agent | `QuantumEFEAgent` |
| Interaction | Coupling $J_{ij}$ | Shared history | Communication | `networks/topology` |
| Conflict | Anti-ferromagnetic $J < 0$ | Everyone can't be minority | Conflicting objectives | `annealing/qubo` |
| Noise | Temperature $T$ | Strategy randomness | Bounded rationality | `quantumness` |
| Quantum | Transverse field $\Gamma$ | Coherent strategies | Quantum tunneling | `models/alf_bridge` |
| Order param | Magnetisation $|m|$ | Volatility $\sigma^2/N$ | Agreement distance | `networks/observables` |
| Transition | $T_c \approx 2.27$ | $\alpha_c \approx 0.34$ | $\Omega(MN^2)$ bound | — |

---

## Exercises

### Basic

**Exercise 5.1:** Run the 2D Ising model at $T = T_c = 2.269$ with L = 10, 20, 40. Plot the magnetisation time series. At criticality, you should see large fluctuations (critical slowing down). How does the autocorrelation time scale with L?

**Exercise 5.2:** Add an external field $h = 0.5$ to the Ising model and re-run the phase diagram. How does the field break the phase transition? (Hint: it shouldn't — there's a crossover instead.)

**Exercise 5.3:** Run the agreement simulation with `frustration=0.5` and sweep `quantumness` from 0 to 0.8. At what quantumness does the system reach ε-agreement fastest?

### Stretch

**Exercise 5.4 (Spin glass):** Modify `_generate_couplings` to draw $J_{ij}$ from a Gaussian distribution $\mathcal{N}(0, 1/N)$ instead of ±1. This is the Sherrington-Kirkpatrick spin glass. Show that it has a glass transition where ergodicity breaks — the system gets trapped in metastable states. Does quantum tunneling help escape?

**Exercise 5.5 (Scaling collapse):** The Ising model near $T_c$ obeys finite-size scaling: $|m| \sim L^{-\beta/\nu} f((T - T_c) L^{1/\nu})$ with $\beta = 1/8$, $\nu = 1$ (2D). Run simulations at L = 10, 20, 40, 80 and attempt a scaling collapse. This is the standard test for universality — the same critical exponents appear in neural criticality models.

## Further Reading

- **Sethna (2021)** — *Statistical Mechanics: Entropy, Order Parameters, and Complexity* (free online, excellent for non-physicists)
- **Brock & Durlauf (2001)** — "Discrete Choice with Social Interactions" (*Review of Economic Studies*) — the Ising-economics connection
- **Nayebi (2025)** — "Intrinsic Barriers and Practical Pathways for Human-AI Alignment" (arXiv:2502.05934) — information-theoretic bounds
- **Challet, Marsili & Zhang (2005)** — *Minority Games* (Oxford) — the anti-ferromagnetic game theory
- **Lewis Cole blog** — "Spin Glass Models" series (lewiscoleblog.com) — accessible computational introduction